IMPORTS

WANG'S EXAMPLES

In [ ]:
import os
import os.path as op

import openneuro
from mne.datasets import sample

from mne_bids import (
    BIDSPath,
    find_matching_paths,
    get_entity_vals,
    make_report,
    print_dir_tree,
    read_raw_bids,
)
import os
from os import path
import openneuro
from mne.datasets import sample
import numpy as np
import matplotlib.pyplot as plt
import mne
from mne.viz import plot_topomap

from naplib.io import load_bids

In [ ]:
import mne  # Import MNE

# Download the sample dataset if it is not already downloaded
data_path = mne.datasets.sample.data_path(download=True)

print("MNE sample dataset path:", data_path)

In [ ]:
# Define the dataset ID
dataset = "ds002778"

# Define the root directory for storing the BIDS dataset
bids_root = op.join(op.dirname(sample.data_path()), dataset)

# Create the directory if it doesn't exist
if not op.isdir(bids_root):
    os.makedirs(bids_root)

In [ ]:
from mne_bids import print_dir_tree

print_dir_tree(bids_root, max_depth=4)


In [ ]:
#dataset = 'ds002778'
#subject = 'pd6'

#bids_root = path.join(path.dirname(sample.data_path()), dataset)
#print(bids_root)
#if not path.isdir(bids_root):
    #os.makedirs(bids_root)

#openneuro.download(dataset=dataset, target_dir=bids_root,
                   include=[f'sub-{subject}'])

In [ ]:
from mne_bids import BIDSPath, read_raw_bids

# Example: Load EEG data for one subject (modify subject/session as needed)
subject = "pd6"
session = "off"  # or "on" for ON medication state
datatype = "eeg"
task = "rest"
suffix = "eeg"

bids_path = BIDSPath(root=bids_root, subject=subject, session=session, datatype=datatype, task=task, suffix=suffix)
raw = read_raw_bids(bids_path)


# Print EEG data info
print(raw.info)
raw.plot()

In [ ]:
# We are only interested in the 32-channel EEG data as the responses, so select those channels
resp_channels = ['Fp1','AF3','F7','F3','FC1','FC5','T7','C3','CP1','CP5','P7',
                 'P3','Pz','PO3','O1','Oz','O2','PO4','P4','P8','CP6','CP2',
                 'C4','T8','FC6','FC2','F4','F8','AF4','Fp2','Fz','Cz']

data = load_bids(root=bids_root, subject=subject, datatype='eeg', task='rest', suffix='eeg', session='off', resp_channels=resp_channels)

In [ ]:
# Apply Notch Filter (Remove Line Noise at 50/60 Hz)
# Removes power line interference.
# Explicitly load data into memory
raw.load_data()  # This replaces preload=True

# Print EEG data info
print(raw.info)


raw.notch_filter(freqs=[50, 60])

In [ ]:
print(bids_path)  # Check if path is correct
print(raw)  # Check if data is successfully loaded


In [ ]:
# Keeps 1-40 Hz signals (filters out unwanted frequencies).
raw.filter(l_freq=1.0, h_freq=40.0)

In [ ]:
# Compute and plot PSD (Power Spectral Density)
psd = raw.compute_psd()
psd.plot(amplitude=False)  # Corrected way to plot PSD

# Handle the missing channels by setting them as EOG or other types
mapping = {
    "EXG1": "eog", "EXG2": "eog", "EXG3": "eog", "EXG4": "eog",
    "EXG5": "eog", "EXG6": "eog", "EXG7": "eog", "EXG8": "eog"
}
raw.set_channel_types(mapping)  # Set non-EEG channels as EOG

# Assign the standard 10-20 montage correctly
raw.set_montage("standard_1020", on_missing="ignore")  # Ignore missing channel positions

In [ ]:
# Detect and Remove Eye Blinks & Movement Artifacts (ICA)
# 
from mne.preprocessing import ICA

ica = ICA(n_components=20, random_state=97)
ica.fit(raw)
ica.plot_components()  # Identify eye/motion artifacts
raw = ica.apply(raw)  # Remove artifacts

In [ ]:
# Extract EEG Features
# Computes Alpha/Theta Ratio per electrode.

import numpy as np
import mne

# Compute Power Spectral Density (PSD) using Welch's method
psd = raw.compute_psd(method="welch", fmin=1, fmax=40, n_fft=1024)
psds, freqs = psd.get_data(return_freqs=True)  # Extract PSD values and frequencies

# Define Alpha (8-12 Hz) and Theta (4-7 Hz) bands
alpha_idx = np.logical_and(freqs >= 8, freqs <= 12)
theta_idx = np.logical_and(freqs >= 4, freqs <= 7)

# Compute Alpha/Theta Ratio
alpha_power = psds[:, alpha_idx].mean(axis=1)
theta_power = psds[:, theta_idx].mean(axis=1)
alpha_theta_ratio = np.log(alpha_power / theta_power)  # Log transform

print("Alpha/Theta Ratio per Channel:", alpha_theta_ratio)



In [ ]:
# Visualize the Alpha/Theta Ratio as a Topomap
# Generates a topographic map of Alpha/Theta ratio.

import matplotlib.pyplot as plt
from mne.viz import plot_topomap

fig, ax = plt.subplots()
ax.set_title('PD6 Alpha/Theta Ratio')
plot_topomap(alpha_theta_ratio, raw.info, axes=ax)
plt.show()

In [ ]:
def log_alpha_theta_ratio(response, sfreq):
    '''response should be of shape (time * channels)'''
    # must transpose response for mne function
    alpha_psd, _ = mne.time_frequency.psd_array_welch(response.T, sfreq, fmin=8, fmax=13, verbose=False) # psd is shape (channels * freqs)
    alpha_psd = alpha_psd.mean(-1)
    
    theta_psd, _ = mne.time_frequency.psd_array_welch(response.T, sfreq, fmin=4, fmax=8, verbose=False) # psd is shape (channels * freqs)
    theta_psd = theta_psd.mean(-1)
    
    return np.log(alpha_psd / theta_psd)
    

alpha_theta_ratio = [log_alpha_theta_ratio(trial['resp'], trial['sfreq']) for trial in data]

In [ ]:
# First, we need to set the montage (i.e. the arrangement of electrodes) so that the channels can be plotted properly
# Here, we set it to the standard 10-20 system, but many options are available if the data were recorded in a different
# montage. See https://mne.tools/dev/generated/mne.channels.make_standard_montage.html for details
data.mne_info.set_montage('standard_1020')

fig, ax = plt.subplots()
ax.set_title('Trial 1: Pd6 Alpha/Theta Ratio')
plot_topomap(alpha_theta_ratio[0], data.mne_info, axes=ax)
plt.show()

In [ ]:
import mne

# Define the correct file path
bdf_file = "/Users/uarc/Downloads/ds002778/sub-pd6/ses-off/eeg/sub-pd6_ses-off_task-rest_eeg.bdf"

# Load BDF file
raw = mne.io.read_raw_bdf(bdf_file, preload=True, verbose=True)

# Print basic information
print(raw.info)


In [ ]:
import matplotlib.pyplot as plt

# Compute PSD
raw.plot_psd(fmin=1, fmax=40, average=True)
plt.show()


In [ ]:
dataset = 'ds002778'
subject = 'hc1'

bids_root = path.join(path.dirname(sample.data_path()), dataset)
print(bids_root)
if not path.isdir(bids_root):
    os.makedirs(bids_root)

openneuro.download(dataset=dataset, target_dir=bids_root,
                   include=[f'sub-{subject}'])

In [ ]:
# We are only interested in the 32-channel EEG data as the responses, so select those channels
resp_channels = ['Fp1','AF3','F7','F3','FC1','FC5','T7','C3','CP1','CP5','P7',
                 'P3','Pz','PO3','O1','Oz','O2','PO4','P4','P8','CP6','CP2',
                 'C4','T8','FC6','FC2','F4','F8','AF4','Fp2','Fz','Cz']


data_hc = load_bids(
    root=bids_root, 
    subject="hc1",   # Subject name
    session="hc",    # FIXED: Changed from "off" to "hc"
    datatype="eeg",
    task="rest",     # Ensure this matches the filename
    suffix="eeg",
    resp_channels=resp_channels  
)

In [ ]:
def log_alpha_theta_ratio(response, sfreq):
    '''response should be of shape (time * channels)'''
    # must transpose response for mne function
    alpha_psd, _ = mne.time_frequency.psd_array_welch(response.T, sfreq, fmin=8, fmax=13, verbose=False) # psd is shape (channels * freqs)
    alpha_psd = alpha_psd.mean(-1)
    
    theta_psd, _ = mne.time_frequency.psd_array_welch(response.T, sfreq, fmin=4, fmax=8, verbose=False) # psd is shape (channels * freqs)
    theta_psd = theta_psd.mean(-1)
    
    return np.log(alpha_psd / theta_psd)
    

alpha_theta_ratio_hc = [log_alpha_theta_ratio(trial['resp'], trial['sfreq']) for trial in data_hc]

In [ ]:
# First, we need to set the montage (i.e. the arrangement of electrodes) so that the channels can be plotted properly
# Here, we set it to the standard 10-20 system, but many options are available if the data were recorded in a different
# montage. See https://mne.tools/dev/generated/mne.channels.make_standard_montage.html for details
data.mne_info.set_montage('standard_1020')

fig, ax = plt.subplots()
ax.set_title('HC1 Alpha/Theta Ratio')
plot_topomap(alpha_theta_ratio_hc[0], data.mne_info, axes=ax)
plt.show()
# Print Alpha/Theta Ratio Details
print(f"Alpha/Theta Ratio per channel:\n{alpha_theta_ratio}")
print(f"Average Alpha/Theta Ratio: {np.mean(alpha_theta_ratio):.2f}")

In [ ]:
import mne

# Define file path
bdf_file = "/Users/uarc/Downloads/ds002778/sub-hc1/ses-hc/eeg/sub-hc1_ses-hc_task-rest_eeg.bdf"

# Load the raw EEG data
raw = mne.io.read_raw_bdf(bdf_file, preload=True, verbose=True)

# Apply a notch filter (to remove power line noise at 50/60 Hz)
raw.notch_filter([50, 60], verbose=True)

# Apply a bandpass filter (e.g., 1-40 Hz)
raw.filter(1, 40, fir_design='firwin', verbose=True)


In [ ]:
import matplotlib.pyplot as plt

# Compute PSD
raw.plot_psd(fmin=1, fmax=40, average=True)
plt.show()

IMPORTS

Download EEG data from OpenNeuro

Import data and get auditory spectrogram which will be used as stimulus.


In [ ]:
dataset = 'ds002778'
subject = 'pd12'

bids_root = path.join(path.dirname(sample.data_path()), dataset)
print(bids_root)
if not path.isdir(bids_root):
    os.makedirs(bids_root)

openneuro.download(dataset=dataset, target_dir=bids_root,
                   include=[f'sub-{subject}'])

Read the data into a Data object

In [ ]:
# We are only interested in the 32-channel EEG data as the responses, so select those channels
resp_channels = ['Fp1','AF3','F7','F3','FC1','FC5','T7','C3','CP1','CP5','P7',
                 'P3','Pz','PO3','O1','Oz','O2','PO4','P4','P8','CP6','CP2',
                 'C4','T8','FC6','FC2','F4','F8','AF4','Fp2','Fz','Cz']

data = load_bids(root=bids_root, subject=subject, datatype='eeg', task='rest', suffix='eeg', session='off', resp_channels=resp_channels)

Compute Alpha Theta Ratio

Let's compute the Alpha/Theta Ratio in each channel. We use the log-value so that ratios above 1 are positive and ratios below 1 are negative, which makes the resulting topomap more clear.

In [ ]:
def log_alpha_theta_ratio(response, sfreq):
    '''response should be of shape (time * channels)'''
    # must transpose response for mne function
    alpha_psd, _ = mne.time_frequency.psd_array_welch(response.T, sfreq, fmin=8, fmax=13, verbose=False) # psd is shape (channels * freqs)
    alpha_psd = alpha_psd.mean(-1)
    
    theta_psd, _ = mne.time_frequency.psd_array_welch(response.T, sfreq, fmin=4, fmax=8, verbose=False) # psd is shape (channels * freqs)
    theta_psd = theta_psd.mean(-1)
    
    return np.log(alpha_psd / theta_psd)
    

alpha_theta_ratio = [log_alpha_theta_ratio(trial['resp'], trial['sfreq']) for trial in data]

Visualize Results with MNE

The Data contains the mne_info attribute (data.mne_info) which we can use for plotting
This info is an instance of mne.Info, and it contains measurement information
like channel names, locations, etc, as well as other metadata

In [ ]:
# First, we need to set the montage (i.e. the arrangement of electrodes) so that the channels can be plotted properly
# Here, we set it to the standard 10-20 system, but many options are available if the data were recorded in a different
# montage. See https://mne.tools/dev/generated/mne.channels.make_standard_montage.html for details
data.mne_info.set_montage('standard_1020')

fig, ax = plt.subplots()
ax.set_title('Pd12 Alpha/Theta Ratio')
plot_topomap(alpha_theta_ratio[0], data.mne_info, axes=ax)
plt.show()

In [ ]:
import mne

# Define the correct file path
bdf_file = "/Users/uarc/Downloads/ds002778/sub-pd12/ses-off/eeg/sub-pd12_ses-off_task-rest_eeg.bdf"

# Load BDF file
raw = mne.io.read_raw_bdf(bdf_file, preload=True, verbose=True)

# Print basic information
print(raw.info)


In [ ]:
import mne

# Define file path
bdf_file = "/Users/uarc/Downloads/ds002778/sub-pd12/ses-off/eeg/sub-pd12_ses-off_task-rest_eeg.bdf"

# Load the raw EEG data
raw = mne.io.read_raw_bdf(bdf_file, preload=True, verbose=True)

# Apply a notch filter (to remove power line noise at 50/60 Hz)
raw.notch_filter([50, 60], verbose=True)

# Apply a bandpass filter (e.g., 1-40 Hz)
raw.filter(1, 40, fir_design='firwin', verbose=True)


In [ ]:
import matplotlib.pyplot as plt

# Compute PSD
raw.plot_psd(fmin=1, fmax=40, average=True)
plt.show()


In [ ]:
dataset = 'ds002778'
subject = 'pd22'

bids_root = path.join(path.dirname(sample.data_path()), dataset)
print(bids_root)
if not path.isdir(bids_root):
    os.makedirs(bids_root)

openneuro.download(dataset=dataset, target_dir=bids_root,
                   include=[f'sub-{subject}'])

In [ ]:
# We are only interested in the 32-channel EEG data as the responses, so select those channels
resp_channels = ['Fp1','AF3','F7','F3','FC1','FC5','T7','C3','CP1','CP5','P7',
                 'P3','Pz','PO3','O1','Oz','O2','PO4','P4','P8','CP6','CP2',
                 'C4','T8','FC6','FC2','F4','F8','AF4','Fp2','Fz','Cz']

data = load_bids(root=bids_root, subject=subject, datatype='eeg', task='rest', suffix='eeg', session='off', resp_channels=resp_channels)

In [ ]:
def log_alpha_theta_ratio(response, sfreq):
    '''response should be of shape (time * channels)'''
    # must transpose response for mne function
    alpha_psd, _ = mne.time_frequency.psd_array_welch(response.T, sfreq, fmin=8, fmax=13, verbose=False) # psd is shape (channels * freqs)
    alpha_psd = alpha_psd.mean(-1)
    
    theta_psd, _ = mne.time_frequency.psd_array_welch(response.T, sfreq, fmin=4, fmax=8, verbose=False) # psd is shape (channels * freqs)
    theta_psd = theta_psd.mean(-1)
    
    return np.log(alpha_psd / theta_psd)
    

alpha_theta_ratio = [log_alpha_theta_ratio(trial['resp'], trial['sfreq']) for trial in data]

In [ ]:
# First, we need to set the montage (i.e. the arrangement of electrodes) so that the channels can be plotted properly
# Here, we set it to the standard 10-20 system, but many options are available if the data were recorded in a different
# montage. See https://mne.tools/dev/generated/mne.channels.make_standard_montage.html for details
data.mne_info.set_montage('standard_1020')

fig, ax = plt.subplots()
ax.set_title('Pd22 Alpha/Theta Ratio')
plot_topomap(alpha_theta_ratio[0], data.mne_info, axes=ax)
plt.show()

In [ ]:
import mne

# Define file path
bdf_file = "//Users/uarc/Downloads/ds002778/sub-pd22/ses-off/eeg/sub-pd22_ses-off_task-rest_eeg.bdf"

# Load the raw EEG data
raw = mne.io.read_raw_bdf(bdf_file, preload=True, verbose=True)

# Apply a notch filter (to remove power line noise at 50/60 Hz)
raw.notch_filter([50, 60], verbose=True)

# Apply a bandpass filter (e.g., 1-40 Hz)
raw.filter(1, 40, fir_design='firwin', verbose=True)


In [ ]:
import matplotlib.pyplot as plt

# Compute PSD
raw.plot_psd(fmin=1, fmax=40, average=True)
plt.show()


In [ ]:
dataset = 'ds002778'
subject = 'pd9'

bids_root = path.join(path.dirname(sample.data_path()), dataset)
print(bids_root)
if not path.isdir(bids_root):
    os.makedirs(bids_root)

openneuro.download(dataset=dataset, target_dir=bids_root,
                   include=[f'sub-{subject}'])

In [ ]:
def log_alpha_theta_ratio(response, sfreq):
    '''response should be of shape (time * channels)'''
    # must transpose response for mne function
    alpha_psd, _ = mne.time_frequency.psd_array_welch(response.T, sfreq, fmin=8, fmax=13, verbose=False) # psd is shape (channels * freqs)
    alpha_psd = alpha_psd.mean(-1)
    
    theta_psd, _ = mne.time_frequency.psd_array_welch(response.T, sfreq, fmin=4, fmax=8, verbose=False) # psd is shape (channels * freqs)
    theta_psd = theta_psd.mean(-1)
    
    return np.log(alpha_psd / theta_psd)
    

alpha_theta_ratio = [log_alpha_theta_ratio(trial['resp'], trial['sfreq']) for trial in data]

In [ ]:
# First, we need to set the montage (i.e. the arrangement of electrodes) so that the channels can be plotted properly
# Here, we set it to the standard 10-20 system, but many options are available if the data were recorded in a different
# montage. See https://mne.tools/dev/generated/mne.channels.make_standard_montage.html for details
data.mne_info.set_montage('standard_1020')

fig, ax = plt.subplots()
ax.set_title('Pd9 Alpha/Theta Ratio')
plot_topomap(alpha_theta_ratio[0], data.mne_info, axes=ax)
plt.show()

In [ ]:
import matplotlib.pyplot as plt

# Compute PSD
raw.plot_psd(fmin=1, fmax=40, average=True)
plt.show()


Download Healthy Participant Data

In [ ]:
dataset = 'ds002778'
subject = 'hc24'

bids_root = path.join(path.dirname(sample.data_path()), dataset)
print(bids_root)
if not path.isdir(bids_root):
    os.makedirs(bids_root)

openneuro.download(dataset=dataset, target_dir=bids_root,
                   include=[f'sub-{subject}'])

In [ ]:
# We are only interested in the 32-channel EEG data as the responses, so select those channels
resp_channels = ['Fp1','AF3','F7','F3','FC1','FC5','T7','C3','CP1','CP5','P7',
                 'P3','Pz','PO3','O1','Oz','O2','PO4','P4','P8','CP6','CP2',
                 'C4','T8','FC6','FC2','F4','F8','AF4','Fp2','Fz','Cz']


data_hc = load_bids(
    root=bids_root, 
    subject="hc24",   # Subject name
    session="hc",    # FIXED: Changed from "off" to "hc"
    datatype="eeg",
    task="rest",     # Ensure this matches the filename
    suffix="eeg",
    resp_channels=resp_channels  
)

In [ ]:
def log_alpha_theta_ratio(response, sfreq):
    '''response should be of shape (time * channels)'''
    # must transpose response for mne function
    alpha_psd, _ = mne.time_frequency.psd_array_welch(response.T, sfreq, fmin=8, fmax=13, verbose=False) # psd is shape (channels * freqs)
    alpha_psd = alpha_psd.mean(-1)
    
    theta_psd, _ = mne.time_frequency.psd_array_welch(response.T, sfreq, fmin=4, fmax=8, verbose=False) # psd is shape (channels * freqs)
    theta_psd = theta_psd.mean(-1)
    
    return np.log(alpha_psd / theta_psd)
    

alpha_theta_ratio_hc = [log_alpha_theta_ratio(trial['resp'], trial['sfreq']) for trial in data_hc]

In [ ]:
# First, we need to set the montage (i.e. the arrangement of electrodes) so that the channels can be plotted properly
# Here, we set it to the standard 10-20 system, but many options are available if the data were recorded in a different
# montage. See https://mne.tools/dev/generated/mne.channels.make_standard_montage.html for details
data.mne_info.set_montage('standard_1020')

fig, ax = plt.subplots()
ax.set_title('HC24 Alpha/Theta Ratio')
plot_topomap(alpha_theta_ratio_hc[0], data.mne_info, axes=ax)
plt.show()

In [ ]:
import matplotlib.pyplot as plt

# Compute PSD
raw.plot_psd(fmin=1, fmax=40, average=True)
plt.show()


In [ ]:
dataset = 'ds002778'
subject = 'hc29'

bids_root = path.join(path.dirname(sample.data_path()), dataset)
print(bids_root)
if not path.isdir(bids_root):
    os.makedirs(bids_root)

openneuro.download(dataset=dataset, target_dir=bids_root,
                   include=[f'sub-{subject}'])

In [ ]:
# We are only interested in the 32-channel EEG data as the responses, so select those channels
resp_channels = ['Fp1','AF3','F7','F3','FC1','FC5','T7','C3','CP1','CP5','P7',
                 'P3','Pz','PO3','O1','Oz','O2','PO4','P4','P8','CP6','CP2',
                 'C4','T8','FC6','FC2','F4','F8','AF4','Fp2','Fz','Cz']


data_hc = load_bids(
    root=bids_root, 
    subject="hc29",   # Subject name
    session="hc",    # FIXED: Changed from "off" to "hc"
    datatype="eeg",
    task="rest",     # Ensure this matches the filename
    suffix="eeg",
    resp_channels=resp_channels  
)

In [ ]:
def log_alpha_theta_ratio(response, sfreq):
    '''response should be of shape (time * channels)'''
    # must transpose response for mne function
    alpha_psd, _ = mne.time_frequency.psd_array_welch(response.T, sfreq, fmin=8, fmax=13, verbose=False) # psd is shape (channels * freqs)
    alpha_psd = alpha_psd.mean(-1)
    
    theta_psd, _ = mne.time_frequency.psd_array_welch(response.T, sfreq, fmin=4, fmax=8, verbose=False) # psd is shape (channels * freqs)
    theta_psd = theta_psd.mean(-1)
    
    return np.log(alpha_psd / theta_psd)
    

alpha_theta_ratio_hc = [log_alpha_theta_ratio(trial['resp'], trial['sfreq']) for trial in data_hc]

In [ ]:
# First, we need to set the montage (i.e. the arrangement of electrodes) so that the channels can be plotted properly
# Here, we set it to the standard 10-20 system, but many options are available if the data were recorded in a different
# montage. See https://mne.tools/dev/generated/mne.channels.make_standard_montage.html for details
data.mne_info.set_montage('standard_1020')

fig, ax = plt.subplots()
ax.set_title('HC29 Alpha/Theta Ratio')
plot_topomap(alpha_theta_ratio_hc[0], data.mne_info, axes=ax)
plt.show()

In [ ]:
import matplotlib.pyplot as plt

# Compute PSD
raw.plot_psd(fmin=1, fmax=40, average=True)
plt.show()

In [ ]:
dataset = 'ds002778'
subject = 'hc33'

bids_root = path.join(path.dirname(sample.data_path()), dataset)
print(bids_root)
if not path.isdir(bids_root):
    os.makedirs(bids_root)

openneuro.download(dataset=dataset, target_dir=bids_root,
                   include=[f'sub-{subject}'])

In [ ]:
# We are only interested in the 32-channel EEG data as the responses, so select those channels
resp_channels = ['Fp1','AF3','F7','F3','FC1','FC5','T7','C3','CP1','CP5','P7',
                 'P3','Pz','PO3','O1','Oz','O2','PO4','P4','P8','CP6','CP2',
                 'C4','T8','FC6','FC2','F4','F8','AF4','Fp2','Fz','Cz']


data_hc = load_bids(
    root=bids_root, 
    subject="hc33",   # Subject name
    session="hc",    # FIXED: Changed from "off" to "hc"
    datatype="eeg",
    task="rest",     # Ensure this matches the filename
    suffix="eeg",
    resp_channels=resp_channels  
)

In [ ]:
# First, we need to set the montage (i.e. the arrangement of electrodes) so that the channels can be plotted properly
# Here, we set it to the standard 10-20 system, but many options are available if the data were recorded in a different
# montage. See https://mne.tools/dev/generated/mne.channels.make_standard_montage.html for details
data.mne_info.set_montage('standard_1020')

fig, ax = plt.subplots()
ax.set_title('HC33 Alpha/Theta Ratio')
plot_topomap(alpha_theta_ratio_hc[0], data.mne_info, axes=ax)
plt.show()

In [ ]:
import matplotlib.pyplot as plt

# Compute PSD
raw.plot_psd(fmin=1, fmax=40, average=True)
plt.show()


In [ ]:
dataset = 'ds002778'
subject = 'hc10'

bids_root = path.join(path.dirname(sample.data_path()), dataset)
print(bids_root)
if not path.isdir(bids_root):
    os.makedirs(bids_root)

openneuro.download(dataset=dataset, target_dir=bids_root,
                   include=[f'sub-{subject}'])

In [ ]:
# We are only interested in the 32-channel EEG data as the responses, so select those channels
resp_channels = ['Fp1','AF3','F7','F3','FC1','FC5','T7','C3','CP1','CP5','P7',
                 'P3','Pz','PO3','O1','Oz','O2','PO4','P4','P8','CP6','CP2',
                 'C4','T8','FC6','FC2','F4','F8','AF4','Fp2','Fz','Cz']


data_hc = load_bids(
    root=bids_root, 
    subject="hc10",   # Subject name
    session="hc",    # FIXED: Changed from "off" to "hc"
    datatype="eeg",
    task="rest",     # Ensure this matches the filename
    suffix="eeg",
    resp_channels=resp_channels  
)

In [ ]:
def log_alpha_theta_ratio(response, sfreq):
    '''response should be of shape (time * channels)'''
    # must transpose response for mne function
    alpha_psd, _ = mne.time_frequency.psd_array_welch(response.T, sfreq, fmin=8, fmax=13, verbose=False) # psd is shape (channels * freqs)
    alpha_psd = alpha_psd.mean(-1)
    
    theta_psd, _ = mne.time_frequency.psd_array_welch(response.T, sfreq, fmin=4, fmax=8, verbose=False) # psd is shape (channels * freqs)
    theta_psd = theta_psd.mean(-1)
    
    return np.log(alpha_psd / theta_psd)
    

alpha_theta_ratio_hc = [log_alpha_theta_ratio(trial['resp'], trial['sfreq']) for trial in data_hc]

In [ ]:
# First, we need to set the montage (i.e. the arrangement of electrodes) so that the channels can be plotted properly
# Here, we set it to the standard 10-20 system, but many options are available if the data were recorded in a different
# montage. See https://mne.tools/dev/generated/mne.channels.make_standard_montage.html for details
data.mne_info.set_montage('standard_1020')

fig, ax = plt.subplots()
ax.set_title('HC10 Alpha/Theta Ratio')
plot_topomap(alpha_theta_ratio_hc[0], data.mne_info, axes=ax)
plt.show()

In [ ]:
import matplotlib.pyplot as plt

# Compute PSD
raw.plot_psd(fmin=1, fmax=40, average=True)
plt.show()


In [ ]:
# Load the raw EEG data
raw = mne.io.read_raw_bdf(bdf_file, preload=True, verbose=True)

# Apply a notch filter (to remove power line noise at 50/60 Hz)
raw.notch_filter([50, 60], verbose=True)

# Apply a bandpass filter (e.g., 1-40 Hz)
raw.filter(1, 40, fir_design='firwin', verbose=True)
